In [3]:
import pandas as pd

ANIME_PATH = "../data/raw/animes.csv"
REVIEWS_PATH = "../data/raw/reviews.csv"

anime_df = pd.read_csv(ANIME_PATH)
reviews_df = pd.read_csv(REVIEWS_PATH)

anime_clean = anime_df.drop_duplicates(subset="uid", keep="first").copy()
reviews_clean = reviews_df.drop_duplicates(subset="uid", keep="first").copy()

print("Anime:", len(anime_clean))
print("Reviews:", len(reviews_clean))

Anime: 16216
Reviews: 130519


In [4]:
review_counts=(
    reviews_clean
    .groupby("anime_uid")
    .size()
    .reset_index(name="review_count")
)

In [5]:
review_counts.head()

,anime_uid,review_count
0,1,361
1,5,42
2,6,130
3,7,30
4,8,4


In [6]:
anime_features = anime_clean.merge(
    review_counts,
    left_on="uid",
    right_on="anime_uid",
    how="left"
)

anime_features["review_count"] = anime_features["review_count"].fillna(0)

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

Rows: 16216
Columns: 14


In [7]:
avg_review_score = (
    reviews_clean
    .groupby("anime_uid")["score"]
    .mean()
    .reset_index(name="avg_review_score")
)

avg_review_score.head()

,anime_uid,avg_review_score
0,1,8.972299
1,5,8.261905
2,6,8.384615
3,7,7.133333
4,8,7.000000


In [8]:
anime_features = anime_features.merge(
    avg_review_score,
    left_on="uid",
    right_on="anime_uid",
    how="left"
)

# Anime with no reviews get 0 as average review score.
anime_features["avg_review_score"] = anime_features["avg_review_score"].fillna(0)

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

anime_features[["title", "review_count", "avg_review_score"]].head()

Rows: 16216
Columns: 16


,title,review_count,avg_review_score
0,Haikyuu!! Second Season,52.0,8.788462
1,Shigatsu wa Kimi no Uso,20.0,6.450000
2,Made in Abyss,305.0,8.655738
3,Fullmetal Alchemist: Brotherhood,637.0,9.243328
4,Kizumonogatari III: Reiketsu-hen,28.0,8.500000


In [9]:
score_std = (
    reviews_clean
    .groupby("anime_uid")["score"]
    .std()
    .reset_index(name="score_std")
)

score_std.head()

,anime_uid,score_std
0,1,1.421778
1,5,1.362557
2,6,1.709003
3,7,1.995397
4,8,0.816497


In [10]:
anime_features = anime_features.merge(
    score_std,
    left_on="uid",
    right_on="anime_uid",
    how="left"
)

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

anime_features[["title", "review_count", "avg_review_score", "score_std"]].head()

Rows: 16216
Columns: 18


,title,review_count,avg_review_score,score_std
0,Haikyuu!! Second Season,52.0,8.788462,1.730199
1,Shigatsu wa Kimi no Uso,20.0,6.450000,2.372540
2,Made in Abyss,305.0,8.655738,1.738546
3,Fullmetal Alchemist: Brotherhood,637.0,9.243328,1.412703
4,Kizumonogatari III: Reiketsu-hen,28.0,8.500000,2.081666


In [12]:
anime_features[
    ["review_count", "avg_review_score", "score_std"]
].describe()

,review_count,avg_review_score,score_std
count,16216.000000,16216.000000,5870.000000
mean,8.048779,3.384648,1.798571
std,33.699166,3.594290,0.874860
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.272005
50%,1.000000,1.000000,1.787301
75%,3.000000,7.000000,2.280351
max,1292.000000,10.000000,6.363961


In [13]:
unique_reviewers = (
    reviews_clean
    .groupby("anime_uid")["profile"]
    .nunique()
    .reset_index(name="unique_reviewers")
)

unique_reviewers.head()

,anime_uid,unique_reviewers
0,1,361
1,5,42
2,6,130
3,7,30
4,8,4


In [16]:
print(anime_features.columns.tolist())

['uid', 'title', 'synopsis', 'genre', 'aired', 'episodes', 'members', 'popularity', 'ranked', 'score', 'img_url', 'link', 'anime_uid_x', 'review_count', 'anime_uid_y', 'avg_review_score', 'anime_uid', 'score_std']


In [17]:
# Remove redundant anime_uid columns created by previous merges
anime_features = anime_features.drop(
    columns=["anime_uid_x", "anime_uid_y", "anime_uid"]
)

# Merge unique reviewer counts
anime_features = anime_features.merge(
    unique_reviewers,
    left_on="uid",
    right_on="anime_uid",
    how="left"
)

# Keep only our master UID
anime_features = anime_features.drop(columns=["anime_uid"])

# Anime with no reviews have zero unique reviewers
anime_features["unique_reviewers"] = anime_features["unique_reviewers"].fillna(0)

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

anime_features[["title", "review_count", "unique_reviewers"]].head()

Rows: 16216
Columns: 16


,title,review_count,unique_reviewers
0,Haikyuu!! Second Season,52.0,52.0
1,Shigatsu wa Kimi no Uso,20.0,20.0
2,Made in Abyss,305.0,305.0
3,Fullmetal Alchemist: Brotherhood,637.0,637.0
4,Kizumonogatari III: Reiketsu-hen,28.0,28.0


In [18]:
anime_features["reviews_per_reviewer"] = (
    anime_features["review_count"] /
    anime_features["unique_reviewers"].replace(0, 1)
)

anime_features[
    ["title", "review_count", "unique_reviewers", "reviews_per_reviewer"]
].head()

,title,review_count,unique_reviewers,reviews_per_reviewer
0,Haikyuu!! Second Season,52.0,52.0,1.0
1,Shigatsu wa Kimi no Uso,20.0,20.0,1.0
2,Made in Abyss,305.0,305.0,1.0
3,Fullmetal Alchemist: Brotherhood,637.0,637.0,1.0
4,Kizumonogatari III: Reiketsu-hen,28.0,28.0,1.0


In [19]:
anime_features["reviews_per_reviewer"].describe()

count    16216.000000
mean         0.500308
std          0.500015
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: reviews_per_reviewer, dtype: float64

In [24]:

print(len(anime_features))
print(len(anime_features.columns))

16216
16


In [25]:
anime_features["has_reviews"] = (
    anime_features["review_count"] > 0
).astype(int)

anime_features[
    ["title", "review_count", "avg_review_score", "has_reviews"]
].head()

,title,review_count,avg_review_score,has_reviews
0,Haikyuu!! Second Season,52.0,8.788462,1
1,Shigatsu wa Kimi no Uso,20.0,6.450000,1
2,Made in Abyss,305.0,8.655738,1
3,Fullmetal Alchemist: Brotherhood,637.0,9.243328,1
4,Kizumonogatari III: Reiketsu-hen,28.0,8.500000,1


In [26]:
anime_features["has_reviews"].value_counts()

has_reviews
1    8113
0    8103
Name: count, dtype: int64

In [27]:
import numpy as np

anime_features["log_review_count"] = np.log1p(
    anime_features["review_count"]
)

anime_features[
    ["title", "review_count", "log_review_count"]
].head()

,title,review_count,log_review_count
0,Haikyuu!! Second Season,52.0,3.970292
1,Shigatsu wa Kimi no Uso,20.0,3.044522
2,Made in Abyss,305.0,5.723585
3,Fullmetal Alchemist: Brotherhood,637.0,6.458338
4,Kizumonogatari III: Reiketsu-hen,28.0,3.367296


In [28]:
anime_features["log_review_count"].describe()

count    16216.000000
mean         0.919506
std          1.245513
min          0.000000
25%          0.000000
50%          0.693147
75%          1.386294
max          7.164720
Name: log_review_count, dtype: float64

In [29]:
anime_features["score_reliability"] = np.log1p(
    anime_features["review_count"]
)

anime_features[
    ["title", "review_count", "avg_review_score", "score_reliability"]
].head()

,title,review_count,avg_review_score,score_reliability
0,Haikyuu!! Second Season,52.0,8.788462,3.970292
1,Shigatsu wa Kimi no Uso,20.0,6.450000,3.044522
2,Made in Abyss,305.0,8.655738,5.723585
3,Fullmetal Alchemist: Brotherhood,637.0,9.243328,6.458338
4,Kizumonogatari III: Reiketsu-hen,28.0,8.500000,3.367296


In [30]:
anime_features[
    ["review_count", "log_review_count", "avg_review_score", "score_std", "unique_reviewers"]
].corr()

,review_count,log_review_count,avg_review_score,score_std,unique_reviewers
review_count,1.000000,0.620859,0.262169,0.075172,1.000000
log_review_count,0.620859,1.000000,0.730957,0.174568,0.620859
avg_review_score,0.262169,0.730957,1.000000,-0.318608,0.262169
score_std,0.075172,0.174568,-0.318608,1.000000,0.075172
unique_reviewers,1.000000,0.620859,0.262169,0.075172,1.000000


In [31]:
anime_features = anime_features.drop(columns=["unique_reviewers"])

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

Rows: 16216
Columns: 18


In [32]:
anime_features.columns.tolist()

['uid',
 'title',
 'synopsis',
 'genre',
 'aired',
 'episodes',
 'members',
 'popularity',
 'ranked',
 'score',
 'img_url',
 'link',
 'review_count',
 'avg_review_score',
 'score_std',
 'has_reviews',
 'log_review_count',
 'score_reliability']

In [33]:
anime_features = anime_features.drop(columns=["score_reliability"])

print("Rows:", len(anime_features))
print("Columns:", len(anime_features.columns))

Rows: 16216
Columns: 17


In [34]:
OUTPUT_PATH = "../data/anime_features.csv"

anime_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(anime_features)} rows to {OUTPUT_PATH}")

Saved 16216 rows to ../data/anime_features.csv


In [35]:
anime_features.loc[
    anime_features["has_reviews"] == 0,
    "avg_review_score"
] = np.nan

anime_features[
    ["title", "review_count", "avg_review_score", "has_reviews"]
].head(10)

,title,review_count,avg_review_score,has_reviews
0,Haikyuu!! Second Season,52.0,8.788462,1
1,Shigatsu wa Kimi no Uso,20.0,6.450000,1
2,Made in Abyss,305.0,8.655738,1
3,Fullmetal Alchemist: Brotherhood,637.0,9.243328,1
4,Kizumonogatari III: Reiketsu-hen,28.0,8.500000,1
5,Mob Psycho 100 II,121.0,8.900826,1
6,Sen to Chihiro no Kamikakushi,156.0,9.076923,1
7,Kimetsu no Yaiba,264.0,7.973485,1
8,Owarimonogatari 2nd Season,41.0,9.317073,1
9,Code Geass: Hangyaku no Lelouch R2,260.0,8.719231,1


In [36]:
anime_features["avg_review_score"].isna().sum()

np.int64(8103)

In [37]:
anime_features[
    ["review_count", "score_std"]
].sort_values("review_count").head(10)

,review_count,score_std
16208,0.0,NaN
12267,0.0,NaN
12266,0.0,NaN
12263,0.0,NaN
12261,0.0,NaN
12260,0.0,NaN
12259,0.0,NaN
12258,0.0,NaN
12257,0.0,NaN
12256,0.0,NaN


In [38]:
anime_features[
    ["review_count", "score_std"]
].groupby("review_count").count().head(10)

,score_std
review_count,
0.0,0
1.0,0
2.0,1172
3.0,748
4.0,495
5.0,355
6.0,279
7.0,212
8.0,186


In [39]:
anime_features["score_difference"] = (
    anime_features["avg_review_score"] - anime_features["score"]
)

anime_features[
    ["title", "score", "avg_review_score", "score_difference"]
].head(10)

,title,score,avg_review_score,score_difference
0,Haikyuu!! Second Season,8.82,8.788462,-0.031538
1,Shigatsu wa Kimi no Uso,8.83,6.450000,-2.380000
2,Made in Abyss,8.83,8.655738,-0.174262
3,Fullmetal Alchemist: Brotherhood,9.23,9.243328,0.013328
4,Kizumonogatari III: Reiketsu-hen,8.83,8.500000,-0.330000
5,Mob Psycho 100 II,8.89,8.900826,0.010826
6,Sen to Chihiro no Kamikakushi,8.90,9.076923,0.176923
7,Kimetsu no Yaiba,8.92,7.973485,-0.946515
8,Owarimonogatari 2nd Season,8.93,9.317073,0.387073
9,Code Geass: Hangyaku no Lelouch R2,8.93,8.719231,-0.210769


In [40]:
anime_features["score_difference"].describe()

count    8113.000000
mean        0.035072
std         1.404675
min        -5.720000
25%        -0.724314
50%         0.050000
75%         0.806667
max         5.920000
Name: score_difference, dtype: float64

In [41]:
anime_features[
    ["title", "score", "avg_review_score", "score_difference", "review_count"]
].sort_values("score_difference").head(10)

,title,score,avg_review_score,score_difference,review_count
9108,Nurarihyon no Mago: Sennen Makyou Recaps,6.72,1.0,-5.72,1.0
8668,Phantom: Requiem for the Phantom Picture Drama,6.71,1.0,-5.71,1.0
12223,Telemonster,6.50,1.0,-5.50,1.0
13749,Fushigi na Melmo,6.43,1.0,-5.43,1.0
2215,Last Waltz: Hakudaku Mamire no Natsu Gasshuku,6.42,1.0,-5.42,1.0
13002,Konpeki no Kantai,6.36,1.0,-5.36,1.0
14623,Jinki:Extend - Sorekara,6.26,1.0,-5.26,1.0
15199,Queen's Blade: Rebellion Specials,6.19,1.0,-5.19,1.0
4831,Marco: Haha wo Tazunete Sanzenri,7.16,2.0,-5.16,1.0
1408,Gunslinger Stratos,6.11,1.0,-5.11,1.0


In [42]:
anime_features[
    ["title", "score", "avg_review_score", "score_difference", "review_count"]
].sort_values("score_difference", ascending=False).head(10)

,title,score,avg_review_score,score_difference,review_count
7122,Bulsajo Robot Phoenix King,4.08,10.0,5.92,1.0
12746,Taegeugsonyeon Huin Dogsuli,4.13,10.0,5.87,1.0
7177,Computer Haekjeonham Pokpa Daejakjeon,4.38,10.0,5.62,1.0
7251,Zonmi-chan: Halloween☆Special Movie!,4.57,10.0,5.43,1.0
7029,Omocha Bako Series Dai 3 Wa: Ehon 1936-nen,4.70,10.0,5.30,1.0
7334,Gold Pencil And Alien Boy,4.71,10.0,5.29,1.0
11797,Mr. Deniroo in Henteko Mushi,4.76,10.0,5.24,1.0
7480,Roll,4.88,10.0,5.12,1.0
8712,Gunnm 3D Special,4.89,10.0,5.11,1.0
8667,Tetopettenson,4.94,10.0,5.06,1.0


In [43]:
anime_features["abs_score_difference"] = (
    anime_features["score_difference"].abs()
)

anime_features[
    ["title", "score_difference", "abs_score_difference", "review_count"]
].head(10)

,title,score_difference,abs_score_difference,review_count
0,Haikyuu!! Second Season,-0.031538,0.031538,52.0
1,Shigatsu wa Kimi no Uso,-2.380000,2.380000,20.0
2,Made in Abyss,-0.174262,0.174262,305.0
3,Fullmetal Alchemist: Brotherhood,0.013328,0.013328,637.0
4,Kizumonogatari III: Reiketsu-hen,-0.330000,0.330000,28.0
5,Mob Psycho 100 II,0.010826,0.010826,121.0
6,Sen to Chihiro no Kamikakushi,0.176923,0.176923,156.0
7,Kimetsu no Yaiba,-0.946515,0.946515,264.0
8,Owarimonogatari 2nd Season,0.387073,0.387073,41.0
9,Code Geass: Hangyaku no Lelouch R2,-0.210769,0.210769,260.0


In [44]:
anime_features["abs_score_difference"].describe()

count    8113.000000
mean        1.040182
std         0.944579
min         0.000000
25%         0.350000
50%         0.766667
75%         1.440000
max         5.920000
Name: abs_score_difference, dtype: float64

In [45]:
(anime_features["abs_score_difference"] >= 2).sum()

np.int64(1150)

In [46]:
anime_features[
    ["review_count", "abs_score_difference"]
].corr()

,review_count,abs_score_difference
review_count,1.000000,-0.178323
abs_score_difference,-0.178323,1.000000


In [47]:
anime_features.columns.tolist()

['uid',
 'title',
 'synopsis',
 'genre',
 'aired',
 'episodes',
 'members',
 'popularity',
 'ranked',
 'score',
 'img_url',
 'link',
 'review_count',
 'avg_review_score',
 'score_std',
 'has_reviews',
 'log_review_count',
 'score_difference',
 'abs_score_difference']

In [48]:
OUTPUT_PATH = "../data/anime_features.csv"

anime_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(anime_features)} rows and {len(anime_features.columns)} columns.")

Saved 16216 rows and 19 columns.
